# NLLB LoRA — MỘT adapter chung Hoa ↔ Việt (ZIP mã nguồn riêng trên Drive)
Hai file nằm ở My Drive: `nllb_lora_shared_train.ipynb` và `nllb_lora_colab_source.zip`. Chọn GPU T4/G4/A100 và Run all; cấp quyền Google/Drive khi được hỏi. Notebook tự kiểm SHA-256 của ZIP và từng file, giải nén, lấy quota rồi chạy workflow LoRA.

Một adapter học **cả hai chiều** (913.094 × 2 mẫu mỗi epoch, 3 epoch ≈ 171.207 bước): mỗi mẫu mang mã ngôn ngữ nguồn/đích của nó; khi sinh bản dịch, mô hình chọn ngôn ngữ đích theo token ngôn ngữ nguồn. Validation mỗi 20.000 bước chạy cả hai chiều (chiều Việt → Hoa chấm BLEU `zh` + NFKC); best theo trung bình chrF; chrF từng chiều nằm trong `metrics.json`. Môi trường Colab, LoRA r16/α32 (q,v), dữ liệu đóng băng, checkpoint/resume giữ nguyên.

Giữ `EXPERIMENT_ID=None` để tiếp tục run chung gần nhất; giữ `EVALUATE_TEST=False` đến khi khóa adapter (khi bật, test được chấm riêng từng chiều). Run Hoa → Việt `zhvi-lora-...` và các artifact cũ không bị đụng tới. ZIP chỉ chứa mã, không chứa dữ liệu hoặc checkpoint.


In [ ]:
from google.colab import drive
from pathlib import Path
import importlib.util, json, shutil, subprocess, sys, uuid

drive.mount("/content/drive")
CONFIG_NAME = "nllb_lora_shared.json"  # MỘT adapter chung Hoa↔Việt (dữ liệu hai chiều trộn)
ID_PREFIX = "shared-lora-"
ACTIVE_PREFIX = "active-lora-shared-"
EXPERIMENT_ID = None  # Chỉ dùng ID adapter chung (shared-lora-...); None tiếp tục run chung gần nhất của profile GPU
DRIVE_FREE_GIB = None  # Điền dung lượng tài khoản còn trống từ Google One/Drive
EVALUATE_TEST = False  # Chỉ bật sau khi khóa adapter và cấu hình đánh giá
BASE = Path("/content/drive/MyDrive")
DELIVERY = BASE / "DACNTT-lora-shared-source-20261106"
OUTPUT = BASE / "DACNTT-lora/run-all"

# Read the separate source ZIP from My Drive and verify before extracting.
import zipfile, hashlib
BUNDLE = BASE / "nllb_lora_colab_source.zip"
BUNDLE_SHA256 = "6b0642b48ea9655edcf22bbc829efe49206c37d52de53889af0272a4761a7cac"
if hashlib.sha256(BUNDLE.read_bytes()).hexdigest() != BUNDLE_SHA256:
    raise ValueError("Source ZIP hash mismatch; use the ZIP shipped with this notebook.")
with zipfile.ZipFile(BUNDLE) as bundle:
    manifest = json.loads(bundle.read("SOURCE_MANIFEST.json"))
    for relative, entry in manifest.items():
        target = DELIVERY / relative
        if DELIVERY.resolve() not in target.resolve().parents:
            raise ValueError("Unsafe source path")
        payload = bundle.read(relative)
        if len(payload) != entry["bytes"] or hashlib.sha256(payload).hexdigest() != entry["sha256"]:
            raise ValueError("Source file hash/size mismatch: " + relative)
        if target.exists():
            if target.read_bytes() != payload:
                raise ValueError("Existing source differs; preserved: " + str(target))
        else:
            target.parent.mkdir(parents=True, exist_ok=True)
            with target.open("xb") as stream:
                stream.write(payload)

def load_module(file, name):
    spec = importlib.util.spec_from_file_location(name, file)
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    return module

if DRIVE_FREE_GIB is None:
    from google.colab import auth
    import google.auth
    from googleapiclient.discovery import build
    auth.authenticate_user()
    credentials, _ = google.auth.default()
    quota = build("drive", "v3", credentials=credentials).about().get(fields="storageQuota").execute()["storageQuota"]
    if "limit" not in quota:
        raise ValueError("Drive không trả quota limit; điền DRIVE_FREE_GIB thực tế rồi chạy lại.")
    DRIVE_FREE_GIB = max(0, int(quota["limit"]) - int(quota.get("usage", 0))) / 1024**3
    print("Drive account free GiB:", DRIVE_FREE_GIB)
device = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total",
    "--format=csv,noheader,nounits"], check=True, capture_output=True, text=True).stdout.strip().splitlines()
if len(device) != 1:
    raise RuntimeError("Chọn runtime một GPU T4/G4/A100.")
gpu_name, memory_mib = device[0].rsplit(",", 1)
speed = load_module(DELIVERY / "nllb_lora/train/profile_speed.py", "lora_profile")
profile = speed.select_profile(gpu_name.strip(), int(memory_mib.strip()) * 1024**2)
active = OUTPUT / (ACTIVE_PREFIX + profile["name"].split("-t")[0] + ".json")
previous_id = EXPERIMENT_ID
if previous_id is None and active.is_file():
    previous_id = json.loads(active.read_text())["experiment_id"]
if previous_id and not previous_id.startswith(ID_PREFIX):
    raise ValueError("Chỉ chấp nhận experiment adapter chung (tiền tố " + ID_PREFIX + "); không nạp run một chiều hay run full cũ.")
state_file = OUTPUT / previous_id / "workflow.json" if previous_id else None
saved = json.loads(state_file.read_text()) if state_file and state_file.is_file() else {}
if saved.get("source_archive"):
    source = Path(saved["source_archive"])
    helper = load_module(source / "nllb_lora/train/colab_workflow.py", "lora_source")
else:
    helper = load_module(DELIVERY / "nllb_lora/train/colab_workflow.py", "lora_source")
    snapshot, _ = helper.prepare_source(DELIVERY, "/content/dacntt-lora-delivery")
    source = Path("/content/dacntt-lora-source") / uuid.uuid4().hex[:10]
    shutil.copytree(snapshot, source)
    config_file = source / "configs" / CONFIG_NAME
    config = speed.profile_config(json.loads(config_file.read_text()), profile, full=True)
    helper.write_json(config_file, config)
if json.loads((source / "configs" / CONFIG_NAME).read_text())["model"]["method"] != "lora":
    raise ValueError("Snapshot phải là LoRA; không fallback full fine-tuning.")
ROOT, SOURCE_ID = helper.prepare_source(source, "/content/dacntt-lora-snapshots")
for name in list(sys.modules):
    if name == "nllb_lora" or name.startswith("nllb_lora."):
        del sys.modules[name]
sys.path.insert(0, str(ROOT))
wf = load_module(ROOT / "nllb_lora/train/colab_workflow.py", "lora_workflow")
SETTINGS = {
    "mode": "full", "experiment_id": previous_id or (ID_PREFIX + profile["name"] + "-" + uuid.uuid4().hex[:10]),
    "config": "configs/" + CONFIG_NAME, "require_lora": True,
    "readiness": None, "auto_tune": True, "evaluate_test": EVALUATE_TEST,
    "drive_output": str(BASE / "DACNTT-lora"),
    "drive_bundle": str(BASE / "DACNTT-data/nllb_zh_vi_20261016"),
    "local_work": "/content/dacntt-lora-work", "venv": "/content/.venv-nllb-lora",
    "drive_free_gib": DRIVE_FREE_GIB, "drive_quota_source": "User checked Google Drive account quota"
}
workflow = wf.Workflow(SETTINGS, ROOT, SOURCE_ID)
if EXPERIMENT_ID is None:
    wf.write_json(active, {"experiment_id": workflow.state["experiment_id"]})
print("Adapter chung Hoa↔Việt | LoRA experiment:", workflow.state["experiment_id"])
print("Budget:", workflow.full_budget(), "updates; validation 20000 (hai chiều); recovery save 2000")
print("Results:", workflow.directory)


In [ ]:
workflow.environment()
workflow.prepare()
RESULT = workflow.run()


In [ ]:
print("Validation BLEU:", RESULT["bleu"], "chrF:", RESULT["chrf"])
print("Best adapter:", RESULT["best_model"])
print("Results:", RESULT["results"])
if "test_results" in RESULT:
    print("Locked final test:", RESULT["test_bleu"], RESULT["test_chrf"], RESULT["test_results"])
else:
    print("Test chưa chạy. Xem validation và khóa cấu hình trước khi bật EVALUATE_TEST.")
